In [ ]:
from pathlib import Path

# Move to project root so .env is found before any imports
project_root = (
    Path(__file__).parent.parent if "__file__" in dir() else Path.cwd().parent
)

In [ ]:
import sys

print(sys.executable)  # should point to .venv/bin/python inside the project

]

In [ ]:
import pandas as pd
from think_reason_learn.policy_induction import PolicyInduction, WeightTrainerConfig
from think_reason_learn.core.llms import GoogleChoice

# Log level

In [ ]:
import logging
import sys

logging.basicConfig(
    level=logging.INFO,  # You might want debug or info
    stream=sys.stdout,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)

logging.getLogger("google_genai.models").setLevel(logging.ERROR)
logging.getLogger("google_genai.models").propagate = False

# Data

In [ ]:
person1 = """\
A is a 30-year-old woman living in San Francisco. She studied computer science at \
Stanford and worked for six years as a senior engineer at Google on \
large-scale distributed systems. She recently left to start an AI-powered healthcare \
analytics company and has already raised a $2M seed round from \
well-known Bay Area investors.
"""

person2 = """\
B is a 25-year-old man based in New York City. He graduated with a degree in \
marketing from NYU and has been working as a marketing manager at Apple for \
the past three years. He is trying to launch a social media app. Before apple, \
he was a product manager at Facebook.
"""

person3 = """\
C is stay in Los Angeles. He is a practicing medical doctor at UCLA \
and is working on a remote patient monitoring platform. He has limited \
technical knowledge and no startup experience, relying heavily on contractors \
for development. He is also a big fan of the Lakers.
"""

person4 = """\
D is a 40-year-old man living in Chicago. He studied law at the University of \
Chicago and has built a career as a corporate lawyer specializing in \
mergers and acquisitions. He is exploring a legal-tech startup idea but is \
still working full-time at his law firm and has no technical or entrepreneurial \
background.
"""

person5 = """\
E is a 28-year-old woman in San Francisco. She studied computer engineering at \
UC Berkeley and worked as a software engineer at a YC-backed fintech startup \
that scaled rapidly. She is now building her own fintech product for underbanked \
communities and has early traction with pilot customers in Latin America.
"""

person6 = """\
F is a 32-year-old man based in New York City. He earned his MBA from Columbia \
Business School after working in marketing roles at Apple and Spotify. He is \
now working on a consumer subscription box startup, but customer acquisition costs \
have been high, and he is struggling to attract investors without stronger traction.
"""

person7 = """\
G is a 27-year-old woman living in Austin, Texas. She studied industrial engineering \
at MIT and later worked as a product manager at Amazon, focusing on supply chain \
logistics. She has teamed up with two cofounders from her professional network to \
launch a logistics automation startup and recently joined a prominent accelerator.
"""

person8 = """\
H has worked in 7 companies, in 3 different industries. He is currently a product \
manager at a startup in the fintech industry. He is looking to launch a new \
product in the edutech industry.
"""

In [ ]:
X = pd.DataFrame(
    {
        "data": [
            person1,
            person2,
            person3,
            person4,
            person5,
            person6,
            person7,
            person8,
        ]
    }
)
y = ["YES", "NO", "NO", "YES", "NO", "NO", "YES", "NO"]

# Policy Induction

In [ ]:
config = WeightTrainerConfig(cv_folds=3, penalty="l1")
pi = PolicyInduction(
    gen_llmc=[
        GoogleChoice(model="gemini-3.5-flash"),
    ],
    predict_llmc=[
        GoogleChoice(model="gemini-3.1-flash-lite"),
    ],
    config=config,
    max_samples_as_context=2,
    max_policy_length=5,
)

In [ ]:
instructions_tem = await pi.set_task(
    task_description="Predict if a startup founder will be successful "
    "or fail based on their background.",
)
print(instructions_tem)

In [ ]:
pi = await pi.fit(X, y)

In [ ]:
pi.get_memory()

In [ ]:
async for sample_index, results, final_answer, token_counter in pi.predict(X):
    print(f"Sample {sample_index}, Predict: {final_answer}")

# Saving

In [ ]:
pi.save("example_policy_induction", for_production=False)

# Loading

In [ ]:
loaded_pi = PolicyInduction.load("example_policy_induction")

In [ ]:
X_predict = X
y_predict = y

async for sample_index, results, final_answer, token_counter in loaded_pi.predict(
    X_predict
):
    idx = int(sample_index)
    print(f"Sample {idx}, Predict: {final_answer}, Target: {y_predict[idx]}")